# R4-M1: LR-A and BiGRU-A
Run on a CUDA runtime after syncing the project code, GR3 locked LR and BiGRU bundle files, A manifests, R4-S1 features, and canonical shards to Drive. No test audio or test features are used. Training follows the fixed B-selected recipes; calibration and thresholds are R4-N1.
Choose a persistent RUN_ID below. Keep the same ID to resume a BiGRU run; use a new ID for a new experiment. LR or export partial outputs cannot be overwritten. The project ledger is updated separately after review.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, hashlib, subprocess, tempfile, xml.etree.ElementTree as ET
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project').resolve()  # adjust if needed
SHARDS = ROOT / 'data/processed/cohort_v1'  # adjust to your canonical tar shard folder
RUN_ID = 'r4_m1_a_v1'  # keep the same ID when resuming
RESUME_BIGRU = False  # set True only when an interrupted BiGRU run has latest.pt
if not RUN_ID or any(ch not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for ch in RUN_ID):
    raise ValueError('RUN_ID must contain only letters, numbers, underscores or hyphens')
LR_RUN = ROOT/'runs/r4'/f'{RUN_ID}_lr'
BIGRU_RUN = ROOT/'runs/r4'/f'{RUN_ID}_bigru'
LR_EXPORT = ROOT/'runs/r4'/f'{RUN_ID}_lr_export'
BIGRU_EXPORT = ROOT/'runs/r4'/f'{RUN_ID}_bigru_export'
required = ['configs/r4_m1_recipes.json', 'locks/B_lock_gr3.json', 'reports/r4/r4_s1_colab_20261114T212857Z.json', 'scripts/r4_m1_preflight.py', 'scripts/r4_m1_lr.py', 'scripts/r4_m1_bigru.py', 'scripts/r4_m1_export.py', 'tests/test_r4_m1_data.py', 'tests/test_r4_m1_lr.py', 'tests/test_r4_m1_bigru.py', 'tests/test_r4_m1_export.py', 'tests/test_r4_m1_trainer_integration.py', 'tests/test_r3_bigru_frontend.py', 'tests/test_m1_trainer.py']
missing = [str(ROOT/p) for p in required if not (ROOT/p).is_file()]
if missing: raise FileNotFoundError('Missing project files: ' + ', '.join(missing))
if not SHARDS.is_dir(): raise FileNotFoundError(f'Shard directory not found: {SHARDS}')
import torch, numpy, scipy, sklearn, pyarrow, librosa, soundfile, pytest
if not torch.cuda.is_available(): raise RuntimeError('Select a CUDA GPU runtime before continuing')
print('Project:', ROOT, '\nShards:', SHARDS, '\nGPU:', torch.cuda.get_device_name(0), '\nRun ID:', RUN_ID)
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT/'src') + os.pathsep + env.get('PYTHONPATH', '')
env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
def run_command(args):
    result = subprocess.run([sys.executable, *map(str, args)], cwd=str(ROOT), env=env)
    if result.returncode: raise RuntimeError(f'Command failed with exit code {result.returncode}; inspect output above.')
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

## Tests and real-data preflight
All required tests must pass without skips. The preflight checks A input hashes and scope; audio hashes are checked when decoded during BiGRU training.

In [ ]:
tests = ['test_r4_m1_data.py', 'test_r4_m1_lr.py', 'test_r4_m1_bigru.py', 'test_r4_m1_export.py', 'test_r4_m1_notebook.py', 'test_r4_m1_trainer_integration.py', 'test_r3_bigru_frontend.py', 'test_m1_trainer.py']
with tempfile.TemporaryDirectory(prefix='r4_m1_tests_') as directory:
    junit = Path(directory)/'results.xml'
    run_command(['-m', 'pytest', '-q', '-p', 'no:cacheprovider', '--junitxml', junit, *[ROOT/'tests'/name for name in tests]])
    suites = ET.parse(junit).getroot()
    skipped = sum(int(node.get('skipped', '0')) for node in suites.iter('testsuite'))
    if skipped: raise RuntimeError(f'{skipped} tests skipped; resolve before training')
print('Required tests passed without skips.')

In [ ]:
run_command([ROOT/'scripts/r4_m1_preflight.py', '--root', ROOT, '--shard-dir', SHARDS])

## LR-A fit
A completed fit may be skipped on rerun only after its evidence and output hashes match. A partial or failed directory requires inspection and a new RUN_ID.

In [ ]:
if LR_RUN.exists():
    path = LR_RUN/'evidence.json'
    if not path.is_file(): raise RuntimeError(f'Incomplete LR run: {LR_RUN}; inspect it and use a new RUN_ID')
    e = json.loads(path.read_text())
    if e.get('status') != 'complete' or e.get('task') != 'R4-M1-LR' or e.get('recipe_sha256') != sha(ROOT/'configs/r4_m1_recipes.json'): raise RuntimeError('Existing LR evidence is incompatible')
    for name, expected in e['outputs'].items():
        if sha(LR_RUN/name) != expected: raise RuntimeError(f'LR output hash mismatch: {name}')
    print('Verified completed LR run:', LR_RUN)
else:
    run_command([ROOT/'scripts/r4_m1_lr.py', '--root', ROOT, '--shard-dir', SHARDS, '--output', LR_RUN])

## BiGRU-A fit
The fit can take substantial GPU time. To resume after interruption, keep RUN_ID, set RESUME_BIGRU=True, and rerun the notebook. Resume checks the frozen provenance and checkpoint consistency. GRU dropout means a resumed run is not guaranteed bit-identical.

In [ ]:
if BIGRU_RUN.exists() and not RESUME_BIGRU:
    raise RuntimeError(f'BiGRU directory exists: {BIGRU_RUN}. Inspect it; set RESUME_BIGRU=True to verify and resume, or use a new RUN_ID.')
if RESUME_BIGRU and not (BIGRU_RUN/'latest.pt').is_file():
    raise RuntimeError('Resume requires an existing latest.pt; inspect the partial directory.')
args = [ROOT/'scripts/r4_m1_bigru.py', '--root', ROOT, '--shard-dir', SHARDS, '--output', BIGRU_RUN]
if RESUME_BIGRU: args.append('--resume')
run_command(args)

## Frozen validation exports
Selection parity must pass before calibration and operating scores are generated. Partial exports cannot be overwritten. A completed export can be verified and skipped on rerun.

In [ ]:
for name, training, output in [('lr', LR_RUN, LR_EXPORT), ('bigru', BIGRU_RUN, BIGRU_EXPORT)]:
    if output.exists():
        evidence_path = output/'evidence.json'
        if not evidence_path.is_file(): raise RuntimeError(f'Incomplete export: {output}; inspect it and use a new RUN_ID')
        evidence = json.loads(evidence_path.read_text())
        if evidence.get('status') != 'complete' or evidence.get('model') != name or evidence.get('recipe_sha256') != sha(ROOT/'configs/r4_m1_recipes.json'): raise RuntimeError(f'Incompatible export: {output}')
        train_evidence = training/('evidence.json' if name == 'lr' else 'training_evidence.json')
        if evidence.get('training_evidence_sha256') != sha(train_evidence): raise RuntimeError(f'Training evidence changed: {training}')
        for file, expected in evidence['outputs'].items():
            if sha(output/file) != expected: raise RuntimeError(f'Export output hash mismatch: {file}')
        print('Verified completed export:', output)
    else:
        run_command([ROOT/'scripts/r4_m1_export.py', '--root', ROOT, '--shard-dir', SHARDS, '--run-dir', training, '--output', output, '--model', name])
print('Training and exports complete. Review evidence and update the project ledger in the handoff step.')